# Covertype + TabFM + XAI
**Before running:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all.

## 1. Install

In [ ]:
!git clone -q --depth 1 https://github.com/google-research/tabfm.git
!pip install -q "./tabfm[pytorch]" lime
!wget -q https://archive.ics.uci.edu/static/public/31/covertype.zip && unzip -o -q covertype.zip

## 2. The dataset
Each row is a 30m × 30m patch of forest in Colorado.
**Objective:** predict which of 7 tree types grows there (classification), using map data such as elevation, slope and distance to water.

In [ ]:
import pandas as pd

columns = (["Elevation", "Aspect", "Slope", "Horiz_Dist_Hydrology", "Vert_Dist_Hydrology",
            "Horiz_Dist_Roadways", "Hillshade_9am", "Hillshade_Noon", "Hillshade_3pm",
            "Horiz_Dist_Fire_Points"]
           + [f"Wilderness_{i}" for i in range(1, 5)]
           + [f"Soil_{i}" for i in range(1, 41)]
           + ["Cover_Type"])
df = pd.read_csv("covtype.data.gz", header=None, names=columns)

tree_names = {1: "Spruce/Fir", 2: "Lodgepole Pine", 3: "Ponderosa Pine", 4: "Cottonwood/Willow",
              5: "Aspen", 6: "Douglas-fir", 7: "Krummholz"}
df["Cover_Type"] = df["Cover_Type"].map(tree_names)

print(df.shape)
print(df["Cover_Type"].value_counts())
df.head()

## 3. Train TabFM and check performance
The full dataset is 581k rows, which is too big for TabFM, so we take a 4,000-row sample: 3,000 to train on and 1,000 to test on.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

sample = df.sample(4000, random_state=42)
X = sample.drop(columns="Cover_Type")
y = sample["Cover_Type"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=1000, stratify=y, random_state=42)

In [ ]:
import torch
from tabfm import TabFMClassifier
from tabfm import tabfm_v1_0_0_pytorch as tabfm_v1

tabfm_model = tabfm_v1.load(device="cuda", dtype=torch.float32)   # downloads ~6.5 GB the first time
model = TabFMClassifier(model=tabfm_model, n_estimators=8, cache_context=True)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))

## 4. XAI 1: Permutation feature importance (global)
Shuffle one feature at a time and see how much accuracy drops. A bigger drop means the model relies on that feature more.

In [ ]:
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt

result = permutation_importance(model, X_test[:500], y_test[:500], n_repeats=3, random_state=42)
importance = pd.Series(result.importances_mean, index=X.columns).sort_values()

importance.tail(10).plot.barh(title="Top 10 features (permutation importance)")
plt.xlabel("Drop in accuracy")
plt.show()

## 5. XAI 2: LIME (local)
Explain why the model made its prediction for one specific patch of forest.

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

explainer = LimeTabularExplainer(X_train.values, feature_names=list(X.columns),
                                 class_names=list(model.classes_), mode="classification")

def predict_fn(rows):
    return model.predict_proba(pd.DataFrame(rows, columns=X.columns))

row = X_test.iloc[0]
print("Actual:", y_test.iloc[0], "| Predicted:", model.predict(X_test.iloc[[0]])[0])

explanation = explainer.explain_instance(row.values, predict_fn, num_features=8, top_labels=1)
explanation.as_pyplot_figure(label=explanation.top_labels[0])
plt.show()